In [1]:
from pathlib import Path

from maite_datasets.image_classification import MilitaryVehicles

data_root = Path("./data")

# Download once; subsequent runs read existing disk files.
MilitaryVehicles(root=data_root, image_set="base", download=True)
MilitaryVehicles(root=data_root, image_set="train", as_datamaite=True)

# The export nests images under the split directory.
data_path = data_root / "militaryvehicles_datamaite_train" / "train"
print(f"Reading from {data_path}")

/home/aweng/2033/dataeval-flow/.nox/docs/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reading from data/militaryvehicles_datamaite_train/train


In [2]:
from dataeval_flow import PipelineConfig, run_task
from dataeval_flow.config import (
    HuggingFaceDatasetConfig,
    SourceConfig,
    TaskConfig,
    ViewConfig,
    ViewOperation,
)
from dataeval_flow.config.extractors import BoVWExtractorConfig
from dataeval_flow.workflows.data_cleaning import DataCleaningConfig

cleaning = DataCleaningConfig(
    name="mv_cleaning",
    outlier_method="adaptive",
    outlier_flags=["dimension", "pixel", "visual"],
    duplicate_cluster_algorithm="hdbscan",
    duplicate_merge_near=True,
)

task = TaskConfig(
    name="mv_tuning",
    workflow="mv_cleaning",
    sources="mv_src",
    extractor="bovw_ext",
    matrix={
        "outlier_threshold": [2.5, 3.5, 4.5],
        "duplicate_cluster_sensitivity": [0.5, 2.0, 3.0],
    },
)

config = PipelineConfig(
    # Each run starts from this seed, so no run's clustering depends on the runs before it.
    seed=42,
    datasets=[
        HuggingFaceDatasetConfig(name="mv_train", path=str(data_path), task="image_classification"),
    ],
    views=[
        # Shuffle before limiting to avoid sampling only the first alphabetical classes.
        ViewConfig(
            name="sample1000",
            operations=[
                ViewOperation(type="Shuffle", params={"seed": 42}),
                ViewOperation(type="Limit", params={"size": 1000}),
            ],
        ),
    ],
    sources=[
        SourceConfig(name="mv_src", dataset="mv_train", view="sample1000"),
    ],
    extractors=[
        BoVWExtractorConfig(name="bovw_ext", vocab_size=256, batch_size=32),
    ],
    workflows=[cleaning],
    tasks=[task],
)

In [3]:
result = run_task(task, config, cache_dir=Path("./cache"))
print(type(result).__name__, len(result.runs), "runs")

MatrixResult 9 runs


In [4]:
print(result.report(detailed=False, width=160))


  DATA CLEANING
  Workflow:  mv_cleaning (data-cleaning), matrix of 9 runs
  Timestamp: 2026-10-03T05:06:58.666416+00:00
  Duration:  4.55s
  Source:    mv_src (mv_train[sample1000])
  Model:     bovw_ext (bovw)

  Health: 27 warnings [!!] across 9 runs — review flagged findings

  #  outlier_threshold  duplicate_cluster_sensitivity  Health  Image Outliers          Classwise Outliers       Duplicates               Label Distribution
  -  -----------------  -----------------------------  ------  ----------------------  -----------------------  -----------------------  -----------------------
  1  2.5                0.5                            [!!]    [!!] 157 images         [!!] worst: 2S19 MSTA    [!!] 4 exact (0.4%), 6   [..] 24 classes, 1000
                                                               (15.7%)                 (25.0%), 23/24 classes   near (0.6%)              items, imbalance 3.0:1
                                                                                  

In [5]:
run = result.runs[3]
print(run.number, run.label)
for finding in run.result.findings:
    print(f"{finding.severity:<8} {finding.title:<20} {finding.brief}")

4 outlier_threshold=3.5, duplicate_cluster_sensitivity=0.5
warning  Image Outliers       112 images (11.2%)
warning  Classwise Outliers   worst: Tornado (20.4%), 23/24 classes over 3.0%
warning  Duplicates           4 exact (0.4%), 6 near (0.6%)
info     Label Distribution   24 classes, 1000 items, imbalance 3.0:1


In [6]:
print(run.result.steps["dupes"].output.data())

shape: (5, 7)
┌──────────┬───────┬──────────┬──────────────┬─────────────┬────────────┬───────────────────────┐
│ group_id ┆ level ┆ dup_type ┆ item_indices ┆ methods     ┆ differs_on ┆ annotation_divergence │
│ ---      ┆ ---   ┆ ---      ┆ ---          ┆ ---         ┆ ---        ┆ ---                   │
│ i64      ┆ str   ┆ str      ┆ list[i64]    ┆ list[str]   ┆ list[str]  ┆ f64                   │
╞══════════╪═══════╪══════════╪══════════════╪═════════════╪════════════╪═══════════════════════╡
│ 0        ┆ item  ┆ exact    ┆ [337, 344]   ┆ ["xxhash"]  ┆ null       ┆ null                  │
│ 1        ┆ item  ┆ exact    ┆ [496, 721]   ┆ ["xxhash"]  ┆ null       ┆ null                  │
│ 2        ┆ item  ┆ near     ┆ [253, 855]   ┆ ["dhash",   ┆ null       ┆ null                  │
│          ┆       ┆          ┆              ┆ "phash"]    ┆            ┆                       │
│ 3        ┆ item  ┆ near     ┆ [337, 344]   ┆ ["cluster"] ┆ null       ┆ null                  │
│ 4   